In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import re
import numpy as np
import os
import duckdb

# Saving and filtering data

In [ ]:
writer = None
total_read = 0
total_kept = 0

reader = pd.read_json("../data/Books.jsonl.gz", lines=True, compression="gzip", chunksize=200_000)

for i, chunk in enumerate(reader):
    total_read += len(chunk)
    mask = chunk["timestamp"].between("2018-01-01", "2024-01-01", inclusive="left")
    filtered = chunk.loc[mask]
    total_kept += len(filtered)
    
    if len(filtered) == 0:
        continue
    
    table = pa.Table.from_pandas(filtered, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter("../data/books_18_23.parquet", table.schema)
    writer.write_table(table)
    
    if (i + 1) % 5 == 0:
        print(f"chunk {i+1}: {total_read:,} read, {total_kept:,} kept")

if writer is not None:
    writer.close()

chunk 5: 1,000,000 read, 382,149 kept
chunk 10: 2,000,000 read, 807,483 kept
chunk 15: 3,000,000 read, 1,187,679 kept
chunk 20: 4,000,000 read, 1,586,983 kept
chunk 25: 5,000,000 read, 1,953,508 kept
chunk 30: 6,000,000 read, 2,323,412 kept
chunk 35: 7,000,000 read, 2,759,294 kept
chunk 40: 8,000,000 read, 3,112,597 kept
chunk 45: 9,000,000 read, 3,475,900 kept
chunk 50: 10,000,000 read, 3,826,771 kept
chunk 55: 11,000,000 read, 4,243,635 kept
chunk 60: 12,000,000 read, 4,642,239 kept
chunk 65: 13,000,000 read, 4,997,607 kept
chunk 70: 14,000,000 read, 5,307,214 kept
chunk 75: 15,000,000 read, 5,704,843 kept
chunk 80: 16,000,000 read, 6,103,603 kept
chunk 85: 17,000,000 read, 6,444,865 kept
chunk 90: 18,000,000 read, 6,799,486 kept
chunk 95: 19,000,000 read, 7,135,720 kept
chunk 100: 20,000,000 read, 7,483,925 kept
chunk 105: 21,000,000 read, 7,805,033 kept
chunk 110: 22,000,000 read, 8,147,195 kept
chunk 115: 23,000,000 read, 8,494,138 kept
chunk 120: 24,000,000 read, 8,790,708 kept
c

In [ ]:
print(f"total read: {total_read:,} lines")
print(f"total kept: {total_kept:,} lines")
print(f"output file size: {os.path.getsize("../data/books_18_23.parquet") / 1024**3 :.2f} gb")

total read: 29,475,453 lines
total kept: 10,606,530 lines
output file size: 2.55 gb


In [ ]:
df_reviews = pd.read_parquet("../data/books_18_23.parquet")

In [30]:
unique_items = set(df_reviews["parent_asin"].unique())
print(f"unique books: {len(unique_items):,}")

unique books: 2,235,029


In [ ]:
df_meta_test = pd.read_json("../data/meta_Books.jsonl.gz",lines=True, compression="gzip", nrows=10_000)
df_meta_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   main_category    10000 non-null  str    
 1   title            10000 non-null  str    
 2   subtitle         9572 non-null   str    
 3   author           7182 non-null   object 
 4   average_rating   10000 non-null  float64
 5   rating_number    10000 non-null  int64  
 6   features         10000 non-null  object 
 7   description      10000 non-null  object 
 8   price            9328 non-null   float64
 9   images           10000 non-null  object 
 10  videos           10000 non-null  object 
 11  store            9735 non-null   str    
 12  categories       10000 non-null  object 
 13  details          10000 non-null  object 
 14  parent_asin      10000 non-null  str    
 15  bought_together  0 non-null      float64
dtypes: float64(3), int64(1), object(7), str(5)
memory usage: 2.5+ MB


In [46]:
meta_cols = ["parent_asin", "title", "author", "price", "main_category", "average_rating", "rating_number", "store"]

In [43]:
def clean_price(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):
        return float(x)
    match = re.search(r"\d+\.?\d*", str(x))
    return float(match.group()) if match else np.nan

In [50]:
df_meta_test.head(2)

,main_category,title,subtitle,author,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Books,Chaucer,"Hardcover – Import, January 1, 2004",{'avatar': 'https://m.media-amazon.com/images/...,4.5,29,[],[],8.23,[{'large': 'https://m.media-amazon.com/images/...,[],Peter Ackroyd (Author),"[Books, Literature & Fiction, History & Critic...",{'Publisher': 'Chatto & Windus; First Edition ...,0701169850,NaN
1,Books,Notes from a Kidwatcher,First Edition,{'avatar': 'https://m.media-amazon.com/images/...,5.0,1,[Contains 23 selected articles by this influen...,"[About the Author, SANDRA WILDE, Ph.D., is wid...",3.52,[{'large': 'https://m.media-amazon.com/images/...,[],Sandra Wilde (Editor),"[Books, Reference, Words, Language & Grammar]",{'Publisher': 'Heinemann; First Edition (May 2...,0435088688,NaN


In [ ]:
writer = None
total_read = 0
total_kept = 0

reader = pd.read_json("../data/meta_Books.jsonl.gz", lines=True, compression="gzip",chunksize=100_000)

for i, chunk in enumerate(reader):
    total_read += len(chunk)    
    filtered = chunk[chunk["parent_asin"].isin(unique_items)]
    total_kept += len(filtered)
    
    filtered = filtered[[c for c in meta_cols if c in filtered.columns]].copy()
    filtered['author'] = filtered['author'].astype(str)
    filtered["price"] = filtered["price"].apply(clean_price).astype("float64")
    table = pa.Table.from_pandas(filtered, preserve_index=False)
    
    if writer is None:
        writer = pq.ParquetWriter("../data/meta_books_filtered.parquet", table.schema)
    
    writer.write_table(table)
    if (i + 1) % 10 == 0:
        print(f"chunk {i+1}: {total_read:,} read, {total_kept:,} kept")

if writer is not None:
    writer.close()

chunk 10: 1,000,000 read, 530,927 kept
chunk 20: 2,000,000 read, 1,058,564 kept
chunk 30: 3,000,000 read, 1,584,981 kept
chunk 40: 4,000,000 read, 2,053,810 kept


In [ ]:
print(f"total read: {total_read:,} lines")
print(f"total kept: {total_kept:,} lines")
print(f"output file size: {os.path.getsize("../data/meta_books_filtered.parquet") / 1024**3 :.2f} gb")

total read: 4,448,181 lines
total kept: 2,235,029 lines
output file size: 0.83 gb


# Dataset preparation

In [ ]:
df_reviews = pd.read_parquet('../data/books_18_23.parquet')
df_reviews.head(2)

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,1,Not a watercolor book! Seems like copies imo.,It is definitely not a watercolor book. The p...,[{'small_image_url': 'https://m.media-amazon.c...,B09BGPFTDB,B09BGPFTDB,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2022-01-17 06:06:38.485,0,True
1,5,Updated: after 1st arrived damaged this one is...,Updated: after first book arrived very damaged...,[],0593235657,0593235657,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,2021-12-27 18:26:44.904,1,True


In [4]:
df_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 10606530 entries, 0 to 10606529
Data columns (total 10 columns):
 #   Column             Dtype         
---  ------             -----         
 0   rating             int64         
 1   title              str           
 2   text               str           
 3   images             object        
 4   asin               str           
 5   parent_asin        str           
 6   user_id            str           
 7   timestamp          datetime64[ms]
 8   helpful_vote       int64         
 9   verified_purchase  bool          
dtypes: bool(1), datetime64[ms](1), int64(2), object(1), str(5)
memory usage: 4.6+ GB


In [ ]:
df_meta = pd.read_parquet('../data/meta_books_filtered.parquet')

In [6]:
df_meta.info()

<class 'pandas.DataFrame'>
RangeIndex: 2235029 entries, 0 to 2235028
Data columns (total 8 columns):
 #   Column          Dtype  
---  ------          -----  
 0   parent_asin     str    
 1   title           str    
 2   author          str    
 3   price           float64
 4   main_category   str    
 5   average_rating  float64
 6   rating_number   int64  
 7   store           str    
dtypes: float64(2), int64(1), str(5)
memory usage: 1.5 GB


## Review data agregation

creating table 'user - product'

In [ ]:
con = duckdb.connect()
con.execute("""
create table interactions as
select
    user_id,
    parent_asin,
    avg(rating)::FLOAT as avg_rating,
    max(timestamp) as last_timestamp,
    count(*) as n_reviews,
    sum(helpful_vote) as helpful_vote_total,
    max(verified_purchase::int) as verified_purchase
from read_parquet('../data/books_18_23.parquet')
group by user_id, parent_asin
""")

In [11]:
print(con.execute("select count(*) from interactions").fetchall())

[(10493731,)]


## Review filtration

оставляем пользователей с ≥ 3 взаимодействиями и товары с ≥ 5 взаимодействиями.

In [12]:
con.execute("""
create table interactions_filtered as
with active_users as (
    select user_id from interactions 
    group by user_id 
    having count(*) >= 3
),
popular_items as (
    select parent_asin from interactions 
    group by parent_asin 
    having count(*) >= 5
)
select i.*
from interactions i
join active_users u using (user_id)
join popular_items p using (parent_asin)
""")

print(con.execute("select count(*) from interactions_filtered").fetchall())
print(con.execute("select count(distinct user_id) from interactions_filtered").fetchall())
print(con.execute("select count(distinct parent_asin) from interactions_filtered").fetchall())

[(3991029,)]
[(750119,)]
[(347540,)]


from 10.5m to 4m

## Joining with metadata

In [ ]:
con.execute("""
copy (
    select
        i.user_id,
        i.parent_asin,
        i.avg_rating as rating,
        i.last_timestamp,
        i.n_reviews,
        i.helpful_vote_total,
        i.verified_purchase,
        m.title,
        m.author,
        m.price,
        m.main_category,
        m.average_rating as item_avg_rating,
        m.rating_number as item_rating_count,
        m.store
    from interactions_filtered i
    left join read_parquet('../data/meta_books_filtered.parquet') m on i.parent_asin = m.parent_asin
) to '../data/interactions_full.parquet' (format parquet)
""")

# Time-based split

In [ ]:
print(con.execute("""
    select min(last_timestamp) as first_ts, max(last_timestamp) as last_ts
    from '../data/interactions_full.parquet'""").df())

                 first_ts                 last_ts
0 2018-01-01 00:00:00.623 2023-09-13 22:45:55.547


In [ ]:
con.execute("""
create or replace table interactions_split as
select *,
    case
        when last_timestamp < timestamp '2022-02-01' then 'train'
        when last_timestamp < timestamp '2022-10-01' then 'val'
        else 'test'
    end as split
from '../data/interactions_full.parquet'
""")

In [53]:
print(con.execute("""select count(*) as all_interactions from interactions_split""").df())

   all_interactions
0           3991029


In [54]:
print(con.execute("""
    select split,
           count(*)/3991029*100 as percentage,
           count(*) as n_interactions,
    from interactions_split
    group by split
""").df())

   split  percentage  n_interactions
0  train   80.942183         3230426
1    val   10.262967          409598
2   test    8.794850          351005


после сплита нужно отфильтровать пользователей и товары, которых нет в train. Иначе модель не сможет дать рекомендации на val/test.

In [56]:
con.execute("""
create table train_users as 
    select distinct user_id 
    from interactions_split
    where split='train';

create table train_items as 
    select distinct parent_asin 
    from interactions_split
    where split='train';
""")

In [ ]:
con.execute('''
copy (
    select s.*
    from interactions_split s
    JOIN train_users u using (user_id)
    JOIN train_items i using (parent_asin)
) to '../data/interactions_final.parquet' (format parquet)
''')

In [ ]:
print(con.execute("""select count(*) as all_interactions from '../data/interactions_final.parquet'""").df())

   all_interactions
0           3527385


In [ ]:
print(con.execute("""
    select split,
        count(*)/3527385*100 as percentage,
        count(*) as n_interactions,
        count(distinct user_id) as n_users,
        count(distinct parent_asin) as n_items,
    from '../data/interactions_final.parquet'
    group by split
    order by percentage desc
""").df())

   split  percentage  n_interactions  n_users  n_items
0  train   91.581327         3230426   692133   308122
1    val    5.497812          193929   106180    79878
2   test    2.920861          103030    62319    54173


In [ ]:
con.close()